# **SymptoScan**

### 🐾 AI-based animal disease prediction system that analyzes animal health information and symptoms to support early disease screening.

## 1. Loading the datasets

In this part, we load the original animal disease dataset from Kaggle and also load the generated dataset that we will use later for comparison.

At this stage, we will not combine or change the datasets yet. We just want to check their sizes and understand how much data we have in each one.

The original dataset will be treated as the real data, while the generated dataset will be used separately to investigate whether adding synthetic data changes the model performance.

In [8]:
import warnings
import numpy as np #numpy
import pandas as pd #pandas
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("Dataset/animal_disease_prediction.csv")
df_generated = pd.read_csv("Dataset/generated_dataset.csv")
pd.set_option('display.width', 120)

warnings.filterwarnings('ignore')
print("Original Dataset from Kaggle\n",df.shape)
print("Generated Dataset by AI\n",df_generated.shape)

Original Dataset from Kaggle
 (431, 22)
Generated Dataset by AI
 (10031, 22)


In [9]:
df.info()

# Create a copy 
df_copy = df.copy()
df_generated_copy = df_generated.copy()

<class 'pandas.DataFrame'>
RangeIndex: 431 entries, 0 to 430
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Animal_Type         431 non-null    str    
 1   Breed               431 non-null    str    
 2   Age                 431 non-null    int64  
 3   Gender              431 non-null    str    
 4   Weight              431 non-null    float64
 5   Symptom_1           431 non-null    str    
 6   Symptom_2           431 non-null    str    
 7   Symptom_3           431 non-null    str    
 8   Symptom_4           431 non-null    str    
 9   Duration            431 non-null    str    
 10  Appetite_Loss       431 non-null    str    
 11  Vomiting            431 non-null    str    
 12  Diarrhea            431 non-null    str    
 13  Coughing            431 non-null    str    
 14  Labored_Breathing   431 non-null    str    
 15  Lameness            431 non-null    str    
 16  Skin_Lesions       

## 2. Data Preprocessing

In this part, we clean and prepare the data before training the models.

First, we check for and remove duplicate rows and remove extra spaces. We also check the data types and convert columns to the correct format.

We remove the Breed column because it has many different categories compared with the size of the real dataset. Keeping it could make the model learn very specific breed patterns instead of learning more general disease patterns.

We also check missing values and make sure the numerical and categorical features are in a suitable format for the models.

After cleaning the data, we check the dataset again to make sure the features and target column are ready for the next steps.

In [10]:
# Remove extra-spaces from the beginning and end of column names
df_copy.columns, df_generated_copy.columns = df_copy.columns.str.strip(), df_generated_copy.columns.str.strip()

In [11]:
# How many Animal type + Breed are there
print(df_copy[['Animal_Type','Breed']].nunique())
print("Animal_Type:",df_copy['Animal_Type'].unique())
print("---------")
print("Breed:",df_copy['Breed'].unique())

Animal_Type      8
Breed          120
dtype: int64
Animal_Type: <ArrowStringArray>
['Dog', 'Cat', 'Cow', 'Horse', 'Rabbit', 'Sheep', 'Goat', 'Pig']
Length: 8, dtype: str
---------
Breed: <ArrowStringArray>
[         'Labrador',           'Siamese',          'Holstein',            'Beagle',           'Persian',
      'Thoroughbred',   'German Shepherd',        'Maine Coon',           'Bulldog',            'Jersey',
 ...
        'Andalusian',             'Tunis',          'Red Poll',          'Pietrain',       'English Lop',
         'Blackface',   'Belted Galloway', 'Yorkshire Terrier',       'Rambouillet',     'Chester White']
Length: 120, dtype: str


In [12]:
# Remove breed from the both datasets
df_copy.drop(columns=['Breed'], inplace=True)
df_generated_copy.drop(columns=['Breed'], inplace=True)
print(df_copy.columns)

Index(['Animal_Type', 'Age', 'Gender', 'Weight', 'Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4', 'Duration',
       'Appetite_Loss', 'Vomiting', 'Diarrhea', 'Coughing', 'Labored_Breathing', 'Lameness', 'Skin_Lesions',
       'Nasal_Discharge', 'Eye_Discharge', 'Body_Temperature', 'Heart_Rate', 'Disease_Prediction'],
      dtype='str')


In [13]:
# Check Missing Values
print(df_copy.isnull().sum())

Animal_Type           0
Age                   0
Gender                0
Weight                0
Symptom_1             0
Symptom_2             0
Symptom_3             0
Symptom_4             0
Duration              0
Appetite_Loss         0
Vomiting              0
Diarrhea              0
Coughing              0
Labored_Breathing     0
Lameness              0
Skin_Lesions          0
Nasal_Discharge       0
Eye_Discharge         0
Body_Temperature      0
Heart_Rate            0
Disease_Prediction    0
dtype: int64


In [ ]:
df_copy[df_copy.duplicated]

,Animal_Type,Age,Gender,Weight,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Duration,Appetite_Loss,...,Diarrhea,Coughing,Labored_Breathing,Lameness,Skin_Lesions,Nasal_Discharge,Eye_Discharge,Body_Temperature,Heart_Rate,Disease_Prediction
421,Dog,4,Male,25.0,Fever,Lethargy,Appetite Loss,Vomiting,3 days,Yes,...,No,No,No,No,No,No,No,39.5°C,120,Parvovirus
422,Cat,2,Female,4.5,Coughing,Sneezing,Eye Discharge,Nasal Discharge,1 week,No,...,No,Yes,No,No,No,Yes,Yes,38.9°C,150,Upper Respiratory Infection
423,Cow,3,Female,600.0,Fever,Nasal Discharge,Labored Breathing,Coughing,5 days,Yes,...,No,Yes,Yes,No,No,Yes,No,40.1°C,90,Foot and Mouth Disease
424,Dog,1,Male,10.0,Diarrhea,Vomiting,Lethargy,Appetite Loss,2 days,Yes,...,Yes,No,No,No,No,No,No,39.2°C,130,Gastroenteritis
425,Cat,5,Male,3.8,Lethargy,Appetite Loss,Skin Lesions,No,2 weeks,Yes,...,No,No,No,No,Yes,No,No,38.7°C,160,Fungal Infection
426,Horse,6,Female,500.0,Coughing,Labored Breathing,Nasal Discharge,Fever,10 days,Yes,...,No,Yes,Yes,No,No,Yes,No,39.8°C,85,Equine Influenza
427,Dog,3,Female,30.0,Lameness,Fever,Skin Lesions,Lethargy,7 days,Yes,...,No,No,No,Yes,Yes,No,No,39.3°C,110,Lyme Disease
428,Cat,2,Male,6.0,Vomiting,Appetite Loss,Lethargy,Diarrhea,4 days,Yes,...,Yes,No,No,No,No,No,No,39.1°C,140,Intestinal Parasites
429,Dog,5,Male,23.0,Labored Breathing,Coughing,Nasal Discharge,Appetite Loss,6 days,Yes,...,No,Yes,Yes,No,No,Yes,No,40.0°C,115,Canine Distemper
430,Cow,4,Female,580.0,Lethargy,Decreased Milk Yield,Fever,No,8 days,Yes,...,No,No,No,No,No,No,No,39.6°C,70,Mastitis


In [19]:
# Remove Duplicated Rows
df_copy = df_copy.drop_duplicates()
print(df_copy.shape)

(421, 21)


In [20]:
# Fix features' datatype (Symptoms)
def fix_bool_features(df):
    bool_features = df[['Appetite_Loss', 'Vomiting', 'Diarrhea', 'Coughing', 'Labored_Breathing', 'Lameness',
           'Skin_Lesions', 'Nasal_Discharge', 'Eye_Discharge']]

    for feature in bool_features:
        df[feature] = df[feature].str.strip().str.lower().map({'yes': True, 'no': False})
        df.head()

    return df

#Change symptoms type on the both datasets
df_copy = fix_bool_features(df_copy)
df_generated_copy = fix_bool_features(df_generated_copy)


**Analysis of the Symptoms**: We analyze the four text-based symptom columns: Symptom_1 to Symptom_4.

We first check the values in each column to understand what kind of symptoms we have and how often they appear.

From this analysis, we found that some symptoms are already available in the existing Boolean columns, while some have different names but the same meaning. We also found some new symptoms that are not included in the existing Boolean columns.

For example, “Loss Of Appetite”, “Reduced Appetite”, and “Appetite Loss” describe the same symptom. We also found similar cases for reduced milk production, reduced wool production, and swelling.

Because of this, we decided to standardize these symptoms instead of keeping the same information in different forms.

In [ ]:
# Analysis all the Symptoms columns
# Distribution of Symptoms across Symptom_1 to Symptom_4
print(df_copy['Symptom_1'].value_counts(),"\n")
print(df_copy['Symptom_2'].value_counts(),"\n")
print(df_copy['Symptom_3'].value_counts(),"\n")
print(df_copy['Symptom_4'].value_counts(),"\n")

Symptom_1
Coughing                139
Vomiting                 61
Lameness                 54
Lethargy                 50
Sneezing                 33
Nasal Discharge          23
Fever                    14
Diarrhea                 11
Appetite Loss             8
Swollen Joints            8
Eye Discharge             7
Weight Loss               7
Labored Breathing         2
Loss of Appetite          2
Skin Lesions              1
Decreased Milk Yield      1
Name: count, dtype: int64 

Symptom_2
Loss of Appetite           155
Diarrhea                    43
Nasal Discharge             40
Vomiting                    35
Lethargy                    22
Swollen Legs                21
Coughing                    20
Fever                       18
Labored Breathing           12
Appetite Loss               11
Swollen Joints              11
Eye Discharge                9
Dehydration                  6
Sneezing                     5
Weight Loss                  4
Lameness                     3
Decrease

The idea here is to convert the text symptoms into the existing Boolean symptom features so that all symptoms have a consistent format.

First, we clean the text by removing extra spaces and converting everything to lowercase. Then, we create a mapping for symptoms that have different names but the same meaning.

For each row, we check **Symptom_1 to Symptom_4** and mark the corresponding Boolean feature as `True` if the symptom is found. For example, **“Loss of Appetite”, “Reduced Appetite”, and “Appetite Loss”** are all mapped to `Appetite_Loss`.

We also add the new symptoms that we found, such as **Fever, Lethargy, Sneezing, Weight Loss, and Dehydration**.

After mapping the symptoms, we remove the original `Symptom_1` to `Symptom_4` columns because their information has already been converted into the standardized Boolean features. This gives us a cleaner and more consistent set of symptom features for the models.


In [22]:
# Mapping ['Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4'] into the bool features
# Assign all the Symptoms into bool features
symptom_cols = ['Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4']

def standardize_symptoms(df):
    # 1. Clean raw text columns
    for col in symptom_cols:
        df[col] = df[col].astype(str).str.strip().str.lower()

    # 2. Synonym map
    synonym_map = {
        'Appetite_Loss': ['appetite loss', 'loss of appetite', 'reduced appetite', 'decreased appetite'],
        'Reduced_Milk': ['reduced milk production', 'decreased milk yield'],
        'Reduced_Wool': ['reduced wool production', 'reduced wool growth'],
        'Swelling': ['swollen joints', 'swollen legs', 'swelling', 'reduced mobility'],
        'Fever': ['fever'],
        'Lethargy': ['lethargy'],
        'Weight_Loss': ['weight loss'],
        'Dehydration': ['dehydration'],
        'Sneezing': ['sneezing']
    }

    # 3. Create normalized binary symptom columns
    for std_name, variations in synonym_map.items():
        # Check if any variation appears across Symptom_1 to Symptom_4 for each row
        df[std_name] = df[symptom_cols].apply(lambda row: row.isin(variations).any(), axis=1).astype(bool)

    # 4. Drop original symptom columns after normalization
    df.drop(columns=symptom_cols, inplace=True)

    return df

df_copy = standardize_symptoms(df_copy)
df_generated_copy = standardize_symptoms(df_generated_copy)

print(df_copy.columns)

Index(['Animal_Type', 'Age', 'Gender', 'Weight', 'Duration', 'Appetite_Loss', 'Vomiting', 'Diarrhea', 'Coughing',
       'Labored_Breathing', 'Lameness', 'Skin_Lesions', 'Nasal_Discharge', 'Eye_Discharge', 'Body_Temperature',
       'Heart_Rate', 'Disease_Prediction', 'Reduced_Milk', 'Reduced_Wool', 'Swelling', 'Fever', 'Lethargy',
       'Weight_Loss', 'Dehydration', 'Sneezing'],
      dtype='str')


In [23]:
# Clean Body Temperature values and convert to float
def clean_body_temperature(df):
    df['Body_Temperature'] = (
        df['Body_Temperature']
        .astype(str)
        .str.replace('°C', '', regex=False)
        .str.strip()
        .astype(float)
    )
    return df

df_copy = clean_body_temperature(df_copy)
df_generated_copy = clean_body_temperature(df_generated_copy)

df_copy[['Body_Temperature']]


,Body_Temperature
0,39.5
1,38.9
2,40.1
3,39.2
4,38.7
...,...
416,39.2
417,39.2
418,39.1
419,39.4


In [ ]:
#Convert Duration text into numeric days
def duration_to_days(value):
    value = str(value).lower().strip()
    num = int(''.join(filter(str.isdigit, value)))
    if 'week' in value:
        return num * 7
    elif 'month' in value:
        return num * 30
    else:  # days
        return num

# Drop Duration column after creating Duration_Days
def clean_duration(df):
    df['Duration_Days'] = df['Duration'].apply(duration_to_days)
    df.drop(columns=['Duration'], inplace=True)
    return df

df_copy = clean_duration(df_copy)
df_generated_copy = clean_duration(df_generated_copy)